# 7장. RAG 시스템 평가 — 02. RAGAS로 RAG 시스템 평가하기

## 학습 목표

- RAGAS의 4가지 핵심 지표(Context Precision, Context Recall, Faithfulness, Answer Relevancy)를 이해
- 앞 노트북에서 생성한 합성 테스트셋으로 RAG 시스템을 평가
- 지표별 점수를 해석하고 개선 방향을 도출

## 사전 지식

- `01-Test-Dataset-Generator-RAGAS.ipynb` 완료 (테스트셋 `data/attention_synthetic_testset.csv` 생성)
- RAG 시스템 구조 이해 (검색기 + 생성기)

## RAGAS 4가지 지표 한눈에 보기

```mermaid
flowchart TD
    Q[질문] --> R[Retriever<br/>검색기]
    R --> C[검색된 컨텍스트]
    Q --> G[Generator<br/>생성기]
    C --> G
    G --> A[생성된 답변]
    GT[reference<br/>정답] -.-> M1
    GT -.-> M2

    C --> M1[Context Precision<br/>관련 문서가<br/>상위에 있는가?]
    C --> M2[Context Recall<br/>필요한 정보를<br/>모두 찾았는가?]
    A --> M3[Faithfulness<br/>답변이 컨텍스트에<br/>근거하는가?]
    Q --> M4[Answer Relevancy<br/>답변이 질문과<br/>관련 있는가?]
    A --> M3
    A --> M4

    M1 --> S[종합 평가 점수]
    M2 --> S
    M3 --> S
    M4 --> S

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class Q,GT input
    class R,G process
    class C,A process
    class M1,M2,M3,M4 storage
    class S output
```

> **검색(Retrieval) 평가**: Context Precision + Context Recall
> **생성(Generation) 평가**: Faithfulness + Answer Relevancy

> 🎯 **강의 포인트**: RAGAS 4가지 지표는 RAG 시스템의 두 구성 요소를 각각 평가합니다. **검색기(Retriever)** 평가 = Context Precision + Context Recall, **생성기(Generator)** 평가 = Faithfulness + Answer Relevancy. 전체 점수가 낮을 때 어느 부분을 개선해야 할지 이 구분으로 판단합니다.

---

## 환경 설정

In [1]:
# 필요한 패키지 설치
# !pip install -qU langchain ragas faiss-cpu langchain-openai

In [2]:
# DeprecationWarning 억제 (RAGAS 0.4.x 내부 경고)
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

from dotenv import load_dotenv
import os

# 환경변수 로드
load_dotenv()

# macOS에서 FAISS 사용 시 OpenMP 중복 로드 방지
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# LangSmith 프로젝트 설정
os.environ["LANGSMITH_PROJECT"] = "02-Eval-RAGAS"

# RAGAS 버전 확인
import ragas
print(f"RAGAS Version: {ragas.__version__}")

# ---------------------------------------------------
# [LangSmith UI 확인 방법]
# 1. https://smith.langchain.com 접속
# 2. 좌측 Projects 클릭
# 3. "Eval-RAGAS" 프로젝트 선택
# 4. 주안점: Datasets 메뉴 → 평가 실험 결과 비교
# ---------------------------------------------------

RAGAS Version: 0.4.3


---

## 테스트 데이터셋 로드

`01-Test-Dataset-Generator-RAGAS.ipynb`에서 생성한 합성 테스트셋을 불러옵니다.

In [3]:
import os

# 데이터 파일 경로 확인
for _path in ["data/attention_synthetic_testset.csv", "data/attention_paper.pdf"]:
    if not os.path.exists(_path):
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {_path}\n현재 디렉토리: {os.getcwd()}")

In [4]:
import pandas as pd
from datasets import Dataset

# CSV 파일 로드
df = pd.read_csv("data/attention_synthetic_testset.csv")

print(f"테스트 케이스 수: {len(df)}")
print(f"컬럼: {list(df.columns)}")

df.head()

테스트 케이스 수: 12
컬럼: ['user_input', 'reference_contexts', 'reference', 'persona_name', 'query_style', 'query_length', 'synthesizer_name']


,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,What Transformer do?,"['Providedproperattributionisprovided,Googlehe...",The Transformer is a new simple network archit...,AI Research Scientist,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
1,What contribution did Jakob make to the develo...,"['trainingfor3.5daysoneightGPUs,asmallfraction...",Jakob proposed replacing RNNs with self-attent...,Machine Learning Researcher,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
2,How does the Extended Neural GPU contribute to...,['areusedinconjunctionwitharecurrentnetwork.\n...,"The Extended Neural GPU, along with models lik...",AI Research Scientist,WEB_SEARCH_LIKE,LONG,single_hop_specific_query_synthesizer
3,What dot-product attention do in machine learn...,['ScaledDot-ProductAttention Multi-HeadAttenti...,Dot-product attention is a mechanism that comp...,AI Research Scientist,POOR_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
4,Whydidtheauthorschooseasinusoidalversionofposi...,"['<1-hop>\n\nrelativepositions,sinceforanyfixe...",The authors chose the sinusoidal version of po...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer


### reference_contexts 컬럼 변환

CSV로 저장하면 리스트가 문자열로 저장돼요. `ast.literal_eval()`로 다시 리스트로 변환합니다.

> **RAGAS 0.4.x 컬럼명 변경**: 0.1.x의 `contexts` → 0.2.x의 `reference_contexts`, `question` → `user_input`, `ground_truth` → `reference`

> **자주 하는 실수**: CSV에서 로드한 `reference_contexts` 컬럼은 문자열 타입입니다. `ast.literal_eval()`로 변환하지 않으면 RAGAS가 컨텍스트를 한 글자씩 순회해 엉뚱한 점수가 나옵니다. `Dataset.map()`으로 전체 변환을 반드시 수행하세요.

In [5]:
# ---------------------------------------------------
# reference_contexts 컬럼: CSV에서 문자열로 저장된 리스트를 실제 리스트로 복원
# ---------------------------------------------------
import ast

# Dataset으로 변환
# Dataset: HuggingFace datasets 라이브러리의 메모리 내 데이터셋
test_dataset = Dataset.from_pandas(df)

# reference_contexts 컬럼을 문자열에서 리스트로 변환
def convert_to_list(example):
    """문자열로 저장된 리스트를 실제 리스트로 변환"""
    contexts = ast.literal_eval(example["reference_contexts"])
    return {"reference_contexts": contexts}

test_dataset = test_dataset.map(convert_to_list)

print(f"Dataset: {test_dataset}")
print(f"\n첫 번째 케이스의 reference_contexts:")
print(test_dataset[0]["reference_contexts"])

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

Dataset: Dataset({
    features: ['user_input', 'reference_contexts', 'reference', 'persona_name', 'query_style', 'query_length', 'synthesizer_name'],
    num_rows: 12
})

첫 번째 케이스의 reference_contexts:
['Providedproperattributionisprovided,Googleherebygrantspermissionto\nreproducethetablesandfiguresinthispapersolelyforuseinjournalisticor\nscholarlyworks.\nAttention Is All You Need\nAshishVaswani∗ NoamShazeer∗ NikiParmar∗ JakobUszkoreit∗\nGoogleBrain GoogleBrain GoogleResearch GoogleResearch\navaswani@google.com noam@google.com nikip@google.com usz@google.com\nLlionJones∗ AidanN.Gomez∗ † ŁukaszKaiser∗\nGoogleResearch UniversityofToronto GoogleBrain\nllion@google.com aidan@cs.toronto.edu lukaszkaiser@google.com\nIlliaPolosukhin∗ ‡\nillia.polosukhin@gmail.com\nAbstract\nThedominantsequencetransductionmodelsarebasedoncomplexrecurrentor\nconvolutionalneuralnetworksthatincludeanencoderandadecoder. Thebest\nperforming models also connect the encoder and decoder through an attention\nmechanism

---

## RAG 시스템 구축

평가 대상이 되는 RAG 시스템을 구축합니다. Attention 논문을 기반으로 질문에 답변하는 시스템이에요.

In [6]:
# ---------------------------------------------------
# RAG 시스템 구축 (문서 로드 → 분할 → 벡터화 → 체인 조립)
# ---------------------------------------------------
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# 1단계: 문서 로드
# PyMuPDFLoader 사용 — PDFPlumberLoader(01번 노트북)보다 텍스트 추출 속도가 빠르고,
# RAG 검색용으로는 레이아웃 보존보다 빠른 처리가 유리해요.
loader = PyMuPDFLoader("data/attention_paper.pdf")
docs = loader.load()
print(f"문서 로드 완료: {len(docs)} 페이지")

# 2단계: 문서 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=50
)
split_documents = text_splitter.split_documents(docs)
print(f"문서 분할 완료: {len(split_documents)} 청크")

# 3단계: 임베딩 및 벡터 DB 생성
# FAISS: 메타(Facebook AI)에서 만든 고속 유사도 검색 라이브러리
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(
    documents=split_documents,
    embedding=embeddings
)
print("벡터 DB 생성 완료")

# 4단계: Retriever 생성
# as_retriever(): 벡터 DB를 LangChain Retriever 인터페이스로 변환
retriever = vectorstore.as_retriever()

# 5단계: 프롬프트 정의
prompt = PromptTemplate.from_template(
    """You are an assistant for question-answering tasks about the Transformer architecture.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.

#Context:
{context}

#Question:
{question}

#Answer:"""
)

# 6단계: LLM 생성
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 7단계: RAG 체인 생성
# RunnablePassthrough(): 입력을 그대로 다음 단계로 전달
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG 시스템 구축 완료")

문서 로드 완료: 15 페이지
문서 분할 완료: 48 청크
벡터 DB 생성 완료
RAG 시스템 구축 완료


### RAG 시스템 동작 확인

In [7]:
# 테스트 질문으로 RAG 시스템 확인
test_question = "What is the main architecture proposed in this paper?"

# 한국어 테스트 질문으로도 확인해 보세요
# test_question = "이 논문에서 제안하는 주요 아키텍처는 무엇인가요?"
answer = chain.invoke(test_question)

print("=" * 80)
print("❓ 질문:", test_question)
print("=" * 80)
print("💬 답변:")
print(answer)
print("=" * 80)


❓ 질문: What is the main architecture proposed in this paper?
💬 답변:
The main architecture proposed in the paper is the Transformer, which is based solely on attention mechanisms and does not use recurrence or convolutions. It consists of stacked self-attention and point-wise, fully connected layers for both the encoder and decoder.


---

## 테스트 데이터셋에 대한 답변 생성

RAGAS 평가에는 RAG 시스템이 생성한 `answer`가 필요해요. 테스트셋의 모든 질문에 답변을 생성합니다.

In [8]:
# 배치 데이터셋 생성 (0.2.x: question → user_input)
batch_questions = [q for q in test_dataset["user_input"]]

print(f"총 {len(batch_questions)}개의 질문에 대해 답변 생성")
print(f"\n질문 예시:")
for i, q in enumerate(batch_questions[:3], 1):
    print(f"{i}. {q}")

총 12개의 질문에 대해 답변 생성

질문 예시:
1. What Transformer do?
2. What contribution did Jakob make to the development of the Transformer model?
3. How does the Extended Neural GPU contribute to the efficiency of machine learning models compared to traditional recurrent networks?


In [9]:
# ---------------------------------------------------
# 테스트셋 전체 질문에 대해 RAG 답변 배치 생성
# ---------------------------------------------------
# ============================================================
# TODO: chain.batch()로 모든 질문에 답변을 생성하세요
# 힌트: answers = chain.batch(batch_questions)
# 예상 결과: len(answers) == len(batch_questions)
# ============================================================

# batch() 메서드로 모든 질문에 대한 답변 생성
# batch(): 여러 입력을 한 번에 처리 (invoke() 반복 대비 효율적)
answers = chain.batch(batch_questions)

print(f"답변 생성 완료: {len(answers)}개")
print(f"\n답변 예시:")
for i, ans in enumerate(answers[:2], 1):
    print(f"\n{i}. {ans[:200]}...")

답변 생성 완료: 12개

답변 예시:

1. The Transformer is a model architecture that relies entirely on self-attention mechanisms to compute representations of its input and output, without using sequence-aligned recurrent neural networks (...

2. Jakob proposed replacing RNNs with self-attention and started the effort to evaluate this idea in the development of the Transformer model....


### 답변을 데이터셋에 추가

In [10]:
# 'answer' 컬럼이 이미 있으면 제거하고 새로 추가
if "answer" in test_dataset.column_names:
    test_dataset = test_dataset.remove_columns(["answer"])

test_dataset = test_dataset.add_column("answer", answers)

# ---------------------------------------------------
# retrieved_contexts 추가: RAG 검색기가 실제로 찾은 컨텍스트
# ---------------------------------------------------
# RAGAS 0.4.x는 두 종류의 컨텍스트를 구분해요:
#   - reference_contexts: 테스트셋 생성 시 정답 근거로 사용된 컨텍스트 (정답)
#   - retrieved_contexts: RAG 검색기가 실제로 찾아온 컨텍스트 (평가 대상)
# Context Precision/Recall은 이 둘을 비교하여 검색기 성능을 측정합니다.

print("검색기로 각 질문의 컨텍스트 검색 중...")
retrieved_contexts = []
for q in batch_questions:
    # retriever.invoke(): 질문에 대해 유사한 문서 검색
    docs_found = retriever.invoke(q)
    retrieved_contexts.append([doc.page_content for doc in docs_found])

# retrieved_contexts 컬럼 추가
if "retrieved_contexts" in test_dataset.column_names:
    test_dataset = test_dataset.remove_columns(["retrieved_contexts"])
test_dataset = test_dataset.add_column("retrieved_contexts", retrieved_contexts)

print(f"Dataset에 answer + retrieved_contexts 컬럼 추가 완료")
print(f"컬럼: {test_dataset.column_names}")

검색기로 각 질문의 컨텍스트 검색 중...
Dataset에 answer + retrieved_contexts 컬럼 추가 완료
컬럼: ['user_input', 'reference_contexts', 'reference', 'persona_name', 'query_style', 'query_length', 'synthesizer_name', 'answer', 'retrieved_contexts']


---

## RAGAS 4가지 지표 상세 이해

평가를 실행하기 전에 각 지표가 정확히 무엇을 측정하는지 살펴볼게요.

### 지표 1. Context Precision (컨텍스트 정확도)

**검색기(Retriever)가 관련 문서를 상위에 올려두었는가?**

이상적으로는 가장 관련성 높은 문서가 1번에 오고, 덜 관련된 문서가 뒤에 와야 해요. Reranker가 없다면 이 지표가 낮게 나오는 경우가 많습니다.

**계산 방법**:

$$\text{Context Precision@K} = \frac{\sum_{k=1}^{K} (\text{Precision@k} \times v_k)}{\text{Total relevant items in top K}}$$

- `v_k` = k번째 문서의 관련성 (0 또는 1)
- 높을수록 관련 문서가 상위에 잘 배치됨

**예시로 이해하기**:

> **질문**: "Transformer 모델의 핵심 메커니즘은 무엇인가요?"
>
> | 순위 | 검색된 문서 | 관련? |
> |:---:|---|:---:|
> | 1 | "Transformer는 Self-Attention 메커니즘을 핵심으로 사용한다." | O |
> | 2 | "BERT는 Transformer의 인코더 부분만 사용한다." | X |
> | 3 | "Multi-Head Attention은 여러 관점에서 문맥을 파악한다." | O |
>
> **좋은 경우 (점수 높음)**: 위처럼 1번에 정답 문서가 바로 있으면 점수가 높아요.  
> **나쁜 경우 (점수 낮음)**: 관련 문서가 3번, 4번에 밀려나고 1~2번이 엉뚱한 문서이면 점수가 낮아요.
>
> 쉽게 말해, **검색 결과의 "순서"가 얼마나 좋은지** 평가하는 지표예요.  
> 네이버 검색에서 원하는 결과가 1페이지 맨 위에 뜨면 좋고, 3페이지에 묻혀 있으면 나쁜 것과 같은 원리입니다.

**목표 점수**: > 0.8

**점수가 낮을 때**: 임베딩 모델 개선, 청크 크기 최적화, Reranker 추가

> 🔑 **핵심 개념**: Faithfulness는 RAG에서 가장 중요한 지표입니다. 점수가 1.0이 아니라는 것은 LLM이 컨텍스트에 없는 내용을 꾸며냈다는 뜻(할루시네이션)입니다. Faithfulness가 낮으면 프롬프트를 강화하거나 temperature를 낮추는 것이 첫 번째 대응책입니다.

### 지표 2. Context Recall (컨텍스트 재현율)

**정답을 생성하는 데 필요한 모든 정보가 검색되었는가?**

`ground_truth`의 각 주장(claim)이 검색된 컨텍스트에서 확인 가능한지 검사해요. 검색 범위(k)가 좁으면 이 지표가 낮게 나옵니다.

**계산 방법**:

$$\text{Context Recall} = \frac{|\text{정답의 주장 중 컨텍스트로 설명 가능한 것}|}{|\text{정답의 전체 주장}|}$$

**예시로 이해하기**:

> **질문**: "Attention 메커니즘의 구성 요소는?"
>
> **정답(Ground Truth)**: "Attention은 Query, Key, Value 세 가지 요소로 구성된다."
>
> 정답에 포함된 주장 3개: (1) Query 사용 (2) Key 사용 (3) Value 사용
>
> **좋은 경우 (Recall = 1.0)**:  
> 검색된 문서에 Query, Key, Value가 **모두** 언급됨 → 3/3 = 1.0
>
> **나쁜 경우 (Recall = 0.33)**:  
> 검색된 문서에 Query만 언급되고 Key, Value는 빠짐 → 1/3 = 0.33
>
> 쉽게 말해, **정답을 만들기 위한 재료가 빠짐없이 검색되었는지** 확인하는 지표예요.  
> 레시피대로 요리하려는데 재료 3개 중 1개만 마트에서 찾았다면, Recall이 낮은 것과 같습니다.

**목표 점수**: > 0.85

**점수가 낮을 때**: 검색 문서 수(k) 늘리기, 하이브리드 검색(키워드 + 벡터) 적용

### 지표 3. Faithfulness (충실도)

**생성된 답변이 컨텍스트에만 근거하는가? — 할루시네이션(Hallucination) 체크**

LLM이 컨텍스트에 없는 내용을 꾸며내는 것을 Hallucination이라고 해요. Faithfulness는 이를 정량화합니다.

**계산 방법**:

$$\text{Faithfulness} = \frac{|\text{답변의 주장 중 컨텍스트로 뒷받침되는 것}|}{|\text{답변의 전체 주장}|}$$

**예시로 이해하기**:

> **컨텍스트**: "Transformer는 Self-Attention 메커니즘을 사용하며, 인코더-디코더 구조로 되어 있다."
>
> **높은 충실도 답변 (Faithfulness = 1.0)**:  
> "Transformer의 핵심은 Self-Attention이며, 인코더-디코더 구조입니다."  
> → 주장 2개(Self-Attention, 인코더-디코더) 모두 컨텍스트에 있음 → 2/2 = 1.0
>
> **낮은 충실도 답변 (Faithfulness = 0.5)**:  
> "Transformer는 Self-Attention을 사용하며, **CNN보다 10배 빠릅니다**."  
> → "Self-Attention 사용"은 컨텍스트에 있지만, "CNN보다 10배 빠르다"는 **LLM이 지어낸 내용** → 1/2 = 0.5
>
> 쉽게 말해, **답변이 주어진 자료에만 충실한지** 확인하는 지표예요.  
> 시험에서 교과서 내용만 적으면 충실도가 높고, 본인 추측을 섞어 적으면 충실도가 낮은 것과 같습니다.

**목표 점수**: > 0.9

**점수가 낮을 때**: 프롬프트에 "컨텍스트 기반으로만 답변" 강조, temperature 낮추기, 더 강력한 LLM 사용

### 지표 4. Answer Relevancy (답변 관련성)

**생성된 답변이 질문에 직접적으로 답하는가?**

답변으로부터 역으로 질문을 생성한 뒤, 원래 질문과의 임베딩 유사도를 측정해요. 질문과 무관한 내용이 많을수록 낮아집니다.

**계산 방법**:

$$\text{Answer Relevancy} = \frac{1}{N} \sum_{i=1}^N \cos(E_{\text{생성질문}_i}, E_{\text{원래질문}})$$

**예시로 이해하기**:

> **질문**: "Transformer에서 Positional Encoding은 왜 필요한가요?"
>
> **높은 관련성 답변 (점수 높음)**:  
> "Transformer는 순서 정보가 없으므로, Positional Encoding으로 단어의 위치를 알려줍니다."  
> → 이 답변에서 역으로 생성된 질문: "Positional Encoding의 역할은?" → 원래 질문과 **유사** → 점수 높음
>
> **낮은 관련성 답변 (점수 낮음)**:  
> "Transformer는 2017년 Google에서 발표되었고, 자연어 처리에서 혁명을 일으켰으며, BERT와 GPT의 기반이 되었습니다."  
> → 이 답변에서 역으로 생성된 질문: "Transformer는 언제 발표되었나?" → 원래 질문과 **동떨어짐** → 점수 낮음
>
> 쉽게 말해, **질문에 대한 "핀트"가 맞는지** 평가하는 지표예요.  
> "서울 날씨 알려줘"라고 물었는데 "서울은 대한민국의 수도입니다"라고 답하면, 틀린 말은 아니지만 관련성이 낮은 거예요.

**점수가 낮은 이유**: 질문에 엉뚱한 답변, 불필요한 정보 과다 포함, 너무 일반적인 답변

**목표 점수**: > 0.85

**점수가 낮을 때**: 프롬프트 개선, Few-shot 예제 추가

---

> 🔑 **핵심 개념**: RAGAS 평가에서 **reference_contexts**(테스트셋에 포함된 정답 컨텍스트)와 **retrieved_contexts**(RAG 시스템이 실제로 검색한 컨텍스트)는 다릅니다. Context Precision/Recall은 이 둘을 비교해서 검색 품질을 측정합니다. 즉, "정답에 필요한 문서를 잘 찾아오는가?"를 평가하는 것이에요. `answer` 컬럼은 생성 품질(Faithfulness, Answer Relevancy)을, `reference_contexts`는 검색 품질을 측정하는 데 사용됩니다.

## RAGAS 평가 실행

In [11]:
# ---------------------------------------------------
# RAGAS 4가지 지표 평가 실행 (RAGAS 0.4.x API)
# ---------------------------------------------------
# ============================================================
# TODO: evaluate() 함수를 호출해 RAGAS 평가를 실행하세요
# 힌트: evaluate(dataset=test_dataset, metrics=[...], llm=ragas_llm, embeddings=ragas_embeddings)
# 예상 결과: 각 메트릭 점수가 담긴 result 객체 반환
# ============================================================

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# RAGAS 평가용 LLM과 Embeddings 래핑
# (평가에서는 LangchainEmbeddingsWrapper 사용 — 테스트셋 생성과 다름!)
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

print("RAGAS 평가 시작...")
print("=" * 80)

# 평가 실행 — llm과 embeddings를 명시적으로 전달
result = evaluate(
    dataset=test_dataset,
    metrics=[
        context_precision,      # 검색 정확도
        context_recall,         # 검색 재현율
        faithfulness,           # 충실도 (Hallucination 체크)
        answer_relevancy,       # 답변 관련성
    ],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

print("\n평가 완료!")
print("=" * 80)
result

RAGAS 평가 시작...


Evaluating:   0%|          | 0/48 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.



평가 완료!


{'context_precision': 0.8634, 'context_recall': 0.9278, 'faithfulness': 0.7959, 'answer_relevancy': 0.8439}

> 💡 **실무 팁**: 지표 해석 기준 — 0.8 이상이면 양호, 0.6~0.8이면 개선 필요, 0.6 미만이면 구조적 문제가 있습니다. Context Precision이 낮으면 Reranker 추가를, Context Recall이 낮으면 검색 문서 수(k)를 늘리는 것을 먼저 시도하세요.

---

## 평가 결과 분석

In [12]:
# DataFrame으로 변환
result_df = result.to_pandas()

print(f"평가 결과: {len(result_df)} rows")
print(f"컬럼: {list(result_df.columns)}")

result_df.head()


평가 결과: 12 rows
컬럼: ['user_input', 'retrieved_contexts', 'reference_contexts', 'response', 'reference', 'persona_name', 'query_style', 'query_length', 'context_precision', 'context_recall', 'faithfulness', 'answer_relevancy']


,user_input,retrieved_contexts,reference_contexts,response,reference,persona_name,query_style,query_length,context_precision,context_recall,faithfulness,answer_relevancy
0,What Transformer do?,[Figure 1: The Transformer - model architectur...,"[Providedproperattributionisprovided,Googleher...",The Transformer is a model architecture that r...,The Transformer is a new simple network archit...,AI Research Scientist,POOR_GRAMMAR,SHORT,0.805556,1.0,1.000000,0.590369
1,What contribution did Jakob make to the develo...,[entirely. Experiments on two machine translat...,"[trainingfor3.5daysoneightGPUs,asmallfractiono...",Jakob proposed replacing RNNs with self-attent...,Jakob proposed replacing RNNs with self-attent...,Machine Learning Researcher,PERFECT_GRAMMAR,SHORT,0.750000,1.0,1.000000,0.718020
2,How does the Extended Neural GPU contribute to...,[2\nBackground\nThe goal of reducing sequentia...,[areusedinconjunctionwitharecurrentnetwork.\nI...,The Extended Neural GPU contributes to the eff...,"The Extended Neural GPU, along with models lik...",AI Research Scientist,WEB_SEARCH_LIKE,LONG,1.000000,1.0,0.500000,0.908915
3,What dot-product attention do in machine learn...,"[Attention(Q, K, V ) = softmax(QKT\n√dk\n)V\n(...",[ScaledDot-ProductAttention Multi-HeadAttentio...,Dot-product attention in machine learning is a...,Dot-product attention is a mechanism that comp...,AI Research Scientist,POOR_GRAMMAR,MEDIUM,1.000000,1.0,0.666667,0.955431
4,Whydidtheauthorschooseasinusoidalversionofposi...,[during training.\n4\nWhy Self-Attention\nIn t...,"[<1-hop>\n\nrelativepositions,sinceforanyfixed...",The authors chose a sinusoidal version of posi...,The authors chose the sinusoidal version of po...,NaN,NaN,NaN,1.000000,1.0,0.875000,0.731548


### 지표별 점수 분석

> **해석 기준**: 0.8 이상이면 양호, 0.6~0.8이면 개선 필요, 0.6 미만이면 문제 있음

> 🎯 **강의 포인트**: 낮은 점수의 케이스를 직접 보는 것이 중요합니다. 어떤 질문 유형(simple/reasoning/multi_context)에서 실패하는지 패턴을 찾으면 개선 방향이 명확해집니다. 숫자만 보지 말고 실패 케이스를 반드시 분석하세요.

In [13]:
# 4가지 메트릭 점수만 추출
metric_cols = ['context_precision', 'context_recall', 'faithfulness', 'answer_relevancy']
metrics_df = result_df[metric_cols]

print("=" * 80)
print("📊 RAGAS 평가 결과")
print("=" * 80)

# 각 메트릭의 평균 점수
print("\n평균 점수:")
for metric in metric_cols:
    score = metrics_df[metric].mean()
    status = "✅" if score > 0.8 else "⚠️" if score > 0.6 else "❌"
    print(f"{status} {metric:25s}: {score:.3f}")

# 전체 평균
overall_score = metrics_df.mean().mean()
print(f"\n{'=' * 80}")
print(f"🎯 전체 평균 점수: {overall_score:.3f}")
print(f"{'=' * 80}")

metrics_df


📊 RAGAS 평가 결과

평균 점수:
✅ context_precision        : 0.863
✅ context_recall           : 0.928
⚠️ faithfulness             : 0.796
✅ answer_relevancy         : 0.844

🎯 전체 평균 점수: 0.858


,context_precision,context_recall,faithfulness,answer_relevancy
0,0.805556,1.000000,1.000000,0.590369
1,0.750000,1.000000,1.000000,0.718020
2,1.000000,1.000000,0.500000,0.908915
3,1.000000,1.000000,0.666667,0.955431
4,1.000000,1.000000,0.875000,0.731548
5,1.000000,0.800000,0.812500,0.749705
6,0.638889,1.000000,0.888889,0.870944
7,0.583333,1.000000,0.923077,0.946574
8,1.000000,0.666667,0.500000,0.935423
9,0.916667,1.000000,0.538462,0.935366


### 결과 해석 가이드: 이 점수는 무엇을 의미할까요?

위 표에서 각 행은 하나의 질문-답변 쌍이에요. 점수를 읽는 방법을 케이스별로 살펴볼게요.

---

**점수 등급 기준**

| 등급 | 점수 범위 | 의미 |
|------|----------|------|
| **Excellent** | 0.9 ~ 1.0 | 거의 완벽. 프로덕션 배포 가능 수준 |
| **Good** | 0.8 ~ 0.9 | 양호. 소폭 개선 여지 있음 |
| **Needs Work** | 0.6 ~ 0.8 | 개선 필요. 특정 부분에 문제가 있음 |
| **Poor** | 0.0 ~ 0.6 | 심각한 문제. 구조적 개선 필요 |

---

**지표별 해석 방법**

> 아래 예시는 학습 목적의 **가상 시나리오**입니다. 실제 점수는 위 표를 참고하세요. 동일한 코드라도 실행할 때마다 LLM 응답 차이로 점수가 달라질 수 있어요.

**1. Context Precision이 높은 경우 (0.8 이상)**
- "검색기가 가져온 문서 중 관련 있는 것이 상위에 잘 배치되었다"는 뜻이에요.
- 만약 특정 케이스에서 0.5 이하라면 → 관련 없는 문서가 상위에 섞여 있다는 뜻. **Reranker를 추가하면 개선 가능**.

**2. Context Recall이 높은 경우 (0.8 이상)**
- "정답을 만드는 데 필요한 정보의 대부분을 검색기가 찾아왔다"는 뜻이에요.
- 만약 특정 케이스에서 0.6 미만이라면 → 정답에 필요한 정보가 충분히 검색되지 않음. **검색 문서 수(k)를 늘리거나, 하이브리드 검색을 적용하면 개선 가능**.

**3. Faithfulness가 낮은 경우 (0.8 미만)**
- "LLM이 답변에서 주장한 내용 중 일부가 컨텍스트에 근거가 없다"는 뜻이에요.
- **근거 없는 부분은 할루시네이션** — LLM이 컨텍스트에 없는 내용을 꾸며냄.
- 0.5 이하인 케이스 = 답변의 절반 이상이 컨텍스트와 무관한 내용. 프롬프트에 **"반드시 주어진 컨텍스트만 사용하세요"** 같은 제약을 추가하면 개선 가능.
- 실무에서 가장 먼저 개선해야 할 지표예요.

**4. Answer Relevancy가 낮은 경우 (0.8 미만)**
- "답변이 질문의 의도를 충분히 반영하지 못하고 있다"는 뜻이에요.
- 0.6 미만인 케이스 = 질문과 관련 없는 내용이 많이 포함됨. 주로 질문이 모호하거나 검색된 컨텍스트가 엉뚱할 때 발생.
- **프롬프트에 Few-shot 예제를 추가하거나, "질문에 직접 답하세요" 지시를 강화**하면 개선 가능.

---

**종합 진단: 위 결과를 어떻게 읽을까요?**

위 평가 결과에서 4가지 지표의 평균 점수를 기준으로 진단합니다.

```
검색기 (Retriever):  Context Precision + Context Recall
  → 두 지표 모두 0.8 이상이면 양호, 0.6~0.8이면 개선 필요

생성기 (Generator):  Faithfulness + Answer Relevancy
  → 두 지표 모두 0.8 이상이면 양호, 0.6~0.8이면 개선 필요
```

개선이 필요한 경우의 우선순위:
1. **Faithfulness가 낮다면**: 프롬프트 강화 (할루시네이션 방지 지시문 추가)
2. **Answer Relevancy가 낮다면**: 프롬프트에 Few-shot 예제 추가, "질문에 직접 답하세요" 지시 강화
3. **Context Recall이 낮다면**: 검색 문서 수(k) 늘리기, 하이브리드 검색 적용
4. **Context Precision이 낮다면**: Reranker 추가, 임베딩 모델 개선

### 점수 분포 확인

In [14]:
# 각 메트릭의 통계 정보
print("=" * 80)
print("📈 메트릭별 통계")
print("=" * 80)

metrics_df.describe().round(3)


📈 메트릭별 통계


,context_precision,context_recall,faithfulness,answer_relevancy
count,12.000,12.000,12.000,12.000
mean,0.863,0.928,0.796,0.844
std,0.153,0.135,0.195,0.118
min,0.583,0.667,0.500,0.590
25%,0.750,0.950,0.635,0.745
50%,0.917,1.000,0.861,0.890
75%,1.000,1.000,0.942,0.935
max,1.000,1.000,1.000,0.955


### 점수가 낮은 케이스 분석

어떤 질문에서 점수가 낮게 나왔는지 확인하고 개선 방향을 찾아볼게요.

In [15]:
# 각 메트릭별로 가장 낮은 점수를 받은 케이스 찾기
print("=" * 80)
print("개선이 필요한 케이스")
print("=" * 80)

# 컬럼명: RAGAS 0.4.x에서는 user_input, response
question_col = "user_input" if "user_input" in result_df.columns else "question"
answer_col = "response" if "response" in result_df.columns else "answer"

for metric in metric_cols:
    worst_idx = result_df[metric].idxmin()
    worst_score = result_df.loc[worst_idx, metric]
    
    print(f"\n{metric}이 가장 낮은 케이스 (점수: {worst_score:.3f}):")
    print(f"질문: {result_df.loc[worst_idx, question_col][:100]}...")
    print(f"답변: {result_df.loc[worst_idx, answer_col][:100]}...")

개선이 필요한 케이스

context_precision이 가장 낮은 케이스 (점수: 0.583):
질문: What are the differences between Scaled Dot-Product Attention and Additive Attention in terms of per...
답변: The differences between Scaled Dot-Product Attention and Additive Attention in terms of performance ...

context_recall이 가장 낮은 케이스 (점수: 0.667):
질문: How does Multi-Head Attention improve the performance of the Transformer model compared to tradition...
답변: Multi-Head Attention improves the performance of the Transformer model by allowing the model to join...

faithfulness이 가장 낮은 케이스 (점수: 0.500):
질문: How does the Extended Neural GPU contribute to the efficiency of machine learning models compared to...
답변: The Extended Neural GPU contributes to the efficiency of machine learning models by using convolutio...

answer_relevancy이 가장 낮은 케이스 (점수: 0.590):
질문: What Transformer do?...
답변: The Transformer is a model architecture that relies entirely on self-attention mechanisms to compute...


---

## 평가 결과 저장

In [16]:
# CSV 파일로 저장
output_path = "data/attention_evaluation_result.csv"
result_df.to_csv(output_path, index=False)

print(f"✅ 평가 결과가 저장되었습니다: {output_path}")
print(f"총 {len(result_df)}개의 평가 케이스")


✅ 평가 결과가 저장되었습니다: data/attention_evaluation_result.csv
총 12개의 평가 케이스


---

## 정리

### RAGAS 4가지 지표 요약

| 지표 | 측정 대상 | 낮을 때 원인 | 개선 방법 |
|------|----------|------------|---------|
| **Context Precision** | 관련 문서의 상위 배치 | 임베딩 품질 부족, Reranker 없음 | 임베딩 모델 개선, Reranker 추가 |
| **Context Recall** | 필요 정보의 검색 완성도 | k 값 작음, 청크 분리 | k 증가, 하이브리드 검색 |
| **Faithfulness** | 답변의 컨텍스트 근거율 | Hallucination 발생 | 프롬프트 강화, temperature 0 |
| **Answer Relevancy** | 답변과 질문의 관련성 | 엉뚱한 답변, 과도한 부가 내용 | 프롬프트 개선, Few-shot 추가 |

### RAGAS vs LangSmith 비교

RAGAS는 로컬에서 빠르게 RAG 파이프라인을 평가할 때 유용해요. 다음 노트북에서 배울 LangSmith는 프로덕션 환경에서의 지속적 모니터링에 특화되어 있습니다.

| 특징 | RAGAS | LangSmith |
|------|-------|-----------|
| 초점 | RAG 특화 4가지 지표 | 범용 평가 + 모니터링 |
| 데이터 관리 | 로컬 CSV | 클라우드 데이터셋 |
| 시각화 | Python 코드 | 웹 대시보드 |
| 실시간 추적 | 없음 | 있음 |

### 다음 노트북 예고

다음에는 **LangSmith**를 활용한 평가 방법을 배워요. 데이터셋을 클라우드에 저장하고, LLM 자체를 평가자로 쓰는 **LLM-as-Judge** 패턴, 그리고 임베딩 거리 기반 평가를 다뤄볼게요.